# Evaluate Kaggle Model Checkpoints

This notebook loads ResNet-UNet and TransUNet checkpoints from the contrastive sweep, evaluates them on the Synapse test set, and writes comparison tables/plots.

By default it pulls the latest source from GitHub. Attach the Synapse dataset and make sure Kaggle can read your Kaggle Model variations.

If Kaggle says `New Models cannot be attached in non-interactive sessions`, manually Add Input for the Kaggle Model variations; the notebook will search checkpoints under `/kaggle/input`.

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

FALL_BACK_URL = "https://github.com/Tommyhuy1705/Automated_Abdominal_Multi-Organ_Segmentation_via_Contrastive_Learning_and_Attention_Mechanisms.git"
GITHUB_REPO_URL = os.getenv("GITHUB_REPO_URL", "").strip() or FALL_BACK_URL
GITHUB_BRANCH = os.getenv("GITHUB_BRANCH", "").strip()

REQUIRED_PACKAGES = {
    "kagglehub": "kagglehub",
    "timm": "timm",
    "scipy": "scipy",
    "pandas": "pandas",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
}

for import_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


def clone_or_pull_repo(repo_url, branch=""):
    repo_dir = Path("/kaggle/working/project_repo")
    if (repo_dir / ".git").exists():
        subprocess.check_call(["git", "-C", str(repo_dir), "fetch", "origin"])
        if branch:
            subprocess.check_call(["git", "-C", str(repo_dir), "checkout", branch])
            subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", branch])
        else:
            subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    elif not repo_dir.exists():
        clone_cmd = ["git", "clone", "--depth", "1"]
        if branch:
            clone_cmd.extend(["--branch", branch])
        clone_cmd.extend([repo_url, str(repo_dir)])
        subprocess.check_call(clone_cmd)
    if not (repo_dir / "src").exists():
        raise FileNotFoundError(f"Repository was cloned, but src/ was not found: {repo_dir}")
    return repo_dir


def find_project_root():
    if GITHUB_REPO_URL:
        return clone_or_pull_repo(GITHUB_REPO_URL, GITHUB_BRANCH)
    candidates = []
    env_root = os.getenv("PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path("/kaggle/working")])
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(sorted(input_root.glob("*")))
    for candidate in candidates:
        if (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục src. Hãy set PROJECT_ROOT hoặc GITHUB_REPO_URL.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

In [ ]:
import json
import warnings

import kagglehub
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy import ndimage as ndi
from sklearn.metrics import average_precision_score, auc, precision_recall_curve, roc_curve
from torch.nn import functional as F

from src.models import DiceCrossEntropyLoss, ResNetUNet, TransUNet
from src.utils import (
    build_segmentation_dataloaders,
    count_parameters,
    load_checkpoint,
    resize_logits_to_target,
    set_seed,
    show_loaded_segmentation_samples,
)

set_seed(42)
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=UserWarning)

def find_data_root():
    env_root = os.getenv("DATA_ROOT")
    if env_root and Path(env_root).exists():
        return Path(env_root)
    candidates = [
        Path("/kaggle/input/synapse-processed"),
        PROJECT_ROOT / "Data" / "project_TransUNet" / "data" / "Synapse",
        PROJECT_ROOT / "data" / "Synapse",
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(path for path in input_root.rglob("Synapse") if path.is_dir())
        candidates.extend(path.parent for path in input_root.rglob("train_npz") if path.is_dir())
    for candidate in candidates:
        if (candidate / "train_npz").exists() and (candidate / "test_vol_h5").exists():
            return candidate
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return input_root

def model_config(architecture, variation, weight, checkpoint_name, handle_env=None, dir_env=None):
    default_handle = f"tensura3607/abdominal-multi-organ-segmentation/pyTorch/{variation}"
    return {
        "architecture": architecture,
        "variation": variation,
        "contrastive_weight": weight,
        "contrastive_label": "cw0" if weight == 0 else f"cw{int(round(weight * 1000)):03d}",
        "handle": os.getenv(handle_env or f"{variation.upper().replace('-', '_')}_MODEL_HANDLE", default_handle),
        "model_dir_env_var": dir_env or f"{variation.upper().replace('-', '_')}_MODEL_DIR",
        "checkpoint_name": checkpoint_name,
    }

DATA_ROOT = find_data_root()
NUM_CLASSES = int(os.getenv("NUM_CLASSES", "9"))
IMAGE_SIZE = (224, 224)
BATCH_SIZE = int(os.getenv("EVAL_BATCH_SIZE", os.getenv("BATCH_SIZE", "8")))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "2"))
MAX_ROC_PIXELS = int(os.getenv("MAX_ROC_PIXELS", "250000"))
MAX_SURFACE_SLICES = int(os.getenv("MAX_SURFACE_SLICES", "0"))
MAX_PREDICTION_MODELS = int(os.getenv("MAX_PREDICTION_MODELS", "4"))
HU_WINDOW = (-125.0, 275.0) if os.getenv("USE_HU_WINDOW", "1") == "1" else None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = [
    "Background",
    "Spleen",
    "Right kidney",
    "Left kidney",
    "Gallbladder",
    "Liver",
    "Stomach",
    "Aorta",
    "Pancreas",
]
if len(CLASS_NAMES) != NUM_CLASSES:
    CLASS_NAMES = [f"Class {idx}" for idx in range(NUM_CLASSES)]

MODEL_CONFIGS = {
    "ResNet-UNet cw0": model_config("ResNetUNet", "resnet-unet", 0.0, "best_resnet_unet.pt", "RESNET_UNET_MODEL_HANDLE", "RESNET_UNET_MODEL_DIR"),
    "ResNet-UNet cw001": model_config("ResNetUNet", "resnet-unet-cw001", 0.01, "best_resnet_unet.pt"),
    "ResNet-UNet cw003": model_config("ResNetUNet", "resnet-unet-cw003", 0.03, "best_resnet_unet.pt"),
    "ResNet-UNet cw005": model_config("ResNetUNet", "resnet-unet-cw005", 0.05, "best_resnet_unet.pt"),
    "TransUNet cw0": model_config("TransUNet", "transunet", 0.0, "best_transunet.pt", "TRANSUNET_MODEL_HANDLE", "TRANSUNET_MODEL_DIR"),
    "TransUNet cw001": model_config("TransUNet", "transunet-cw001", 0.01, "best_transunet.pt"),
    "TransUNet cw003": model_config("TransUNet", "transunet-cw003", 0.03, "best_transunet.pt"),
    "TransUNet cw005": model_config("TransUNet", "transunet-cw005", 0.05, "best_transunet.pt"),
}

OUTPUT_DIR = Path("/kaggle/working/evaluation_report") if Path("/kaggle").exists() else PROJECT_ROOT / "artifacts" / "evaluation_report"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT = {DATA_ROOT}")
print(f"DEVICE = {DEVICE}")
print(f"BATCH_SIZE = {BATCH_SIZE}, NUM_WORKERS = {NUM_WORKERS}")
print(f"MAX_ROC_PIXELS = {MAX_ROC_PIXELS}, MAX_SURFACE_SLICES = {MAX_SURFACE_SLICES or 'all'}")
print(f"MAX_PREDICTION_MODELS = {MAX_PREDICTION_MODELS}")
for model_name, cfg in MODEL_CONFIGS.items():
    print(f"{model_name}: {cfg['handle']}")

In [ ]:
loaders, records = build_segmentation_dataloaders(
    data_root=DATA_ROOT,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    hu_window=HU_WINDOW,
)

print({split: len(items) for split, items in records.items()})
print(f"Test batches: {len(loaders['test'])}")
show_loaded_segmentation_samples(loaders["train"], max_samples=3, num_classes=NUM_CLASSES, title="Train samples after loading", prefer_foreground=True)
show_loaded_segmentation_samples(loaders["test"], max_samples=3, num_classes=NUM_CLASSES, title="Test samples after loading", prefer_foreground=True)

In [ ]:
def read_model_info(model_dir):
    matches = [model_dir / "model_info.json"] + sorted(model_dir.rglob("model_info.json"))
    for info_path in matches:
        if info_path.exists():
            return json.loads(info_path.read_text(encoding="utf-8"))
    return {}

def find_checkpoint(model_dir, preferred_name):
    matches = sorted(model_dir.rglob(preferred_name))
    if matches:
        return matches[0]
    pt_files = sorted(model_dir.rglob("*.pt")) + sorted(model_dir.rglob("*.pth"))
    if not pt_files:
        raise FileNotFoundError(f"No .pt/.pth checkpoint found in model folder: {model_dir}")
    return pt_files[0]

def score_candidate_path(candidate, cfg):
    text = str(candidate).lower().replace("-", " ").replace("_", " ")
    parts = [part.lower() for part in cfg["handle"].replace("/", " ").replace("-", " ").replace("_", " ").split()]
    parts.extend(str(cfg.get("variation", "")).lower().replace("-", " ").replace("_", " ").split())
    return sum(part in text for part in parts if part)

def find_attached_kaggle_model_dir(model_name, cfg):
    env_var = cfg["model_dir_env_var"]
    env_dir = os.getenv(env_var, "").strip()
    if env_dir:
        path = Path(env_dir)
        if path.exists():
            print(f"Using {env_var} = {path}")
            return path
        raise FileNotFoundError(f"{env_var} was set, but path does not exist: {path}")

    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None

    preferred_matches = sorted(input_root.rglob(cfg["checkpoint_name"]))
    if preferred_matches:
        scored = sorted(
            ((score_candidate_path(path.parent, cfg), path.parent, path) for path in preferred_matches),
            key=lambda item: (-item[0], len(str(item[1]))),
        )
        _, chosen_dir, checkpoint = scored[0]
        print(f"Found attached checkpoint for {model_name}: {checkpoint}")
        return chosen_dir

    scored_dirs = []
    for candidate in input_root.rglob("*"):
        if not candidate.is_dir():
            continue
        files = list(candidate.glob("*.pt")) + list(candidate.glob("*.pth"))
        if files:
            scored_dirs.append((score_candidate_path(candidate, cfg), candidate))
    if scored_dirs:
        scored_dirs.sort(key=lambda item: (-item[0], len(str(item[1]))))
        chosen = scored_dirs[0][1]
        print(f"Using attached model candidate for {model_name}: {chosen}")
        return chosen
    return None

def resolve_model_dir(model_name, cfg):
    try:
        print(f"Downloading {model_name} with kagglehub: {cfg['handle']}")
        return Path(kagglehub.model_download(cfg["handle"])), "kagglehub"
    except Exception as exc:
        print(f"kagglehub.model_download failed for {model_name}: {type(exc).__name__}: {exc}")
        print("Fallback: searching attached Kaggle inputs under /kaggle/input.")
        model_dir = find_attached_kaggle_model_dir(model_name, cfg)
        if model_dir is None:
            env_var = cfg["model_dir_env_var"]
            raise RuntimeError(
                "Could not download the Kaggle Model in this non-interactive session and no attached checkpoint was found. "
                f"Add the Kaggle Model as an Input for {model_name}, or set {env_var} to the checkpoint folder. "
                f"Handle: {cfg['handle']}"
            ) from exc
        return model_dir, "attached_input"

def build_model(model_name, cfg, model_info):
    architecture = cfg.get("architecture") or model_info.get("architecture")
    if architecture == "ResNetUNet":
        return ResNetUNet(
            num_classes=NUM_CLASSES,
            pretrained_encoder=False,
            decoder_dropout=float(model_info.get("decoder_dropout", 0.1)),
        )
    if architecture == "TransUNet":
        return TransUNet(
            num_classes=NUM_CLASSES,
            pretrained_encoder=False,
            pretrained_transformer=False,
            transformer_model_name=model_info.get("transformer_model_name", "vit_base_patch16_224.augreg_in21k"),
            freeze_first_n_transformer_blocks=0,
            decoder_dropout=float(model_info.get("decoder_dropout", 0.1)),
        )
    raise ValueError(f"Unsupported model architecture for {model_name}: {architecture}")

model_artifacts = {}
for model_name, cfg in MODEL_CONFIGS.items():
    print(f"\nResolving {model_name}: {cfg['handle']}")
    model_dir, source = resolve_model_dir(model_name, cfg)
    model_info = read_model_info(model_dir)
    checkpoint_path = find_checkpoint(model_dir, cfg["checkpoint_name"])
    model_artifacts[model_name] = {
        "handle": cfg["handle"],
        "architecture": cfg["architecture"],
        "variation": cfg["variation"],
        "contrastive_label": cfg["contrastive_label"],
        "contrastive_weight": cfg["contrastive_weight"],
        "source": source,
        "model_dir": str(model_dir),
        "checkpoint_path": str(checkpoint_path),
        "model_info": model_info,
        "checkpoint_keys": [],
    }
    print(f"Resolved checkpoint: {checkpoint_path}")

(OUTPUT_DIR / "model_artifacts.json").write_text(json.dumps(model_artifacts, indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
def safe_divide(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)
    return np.divide(numerator, denominator, out=np.full_like(numerator, np.nan), where=denominator != 0)


def case_id_from_record(record):
    path = Path(record.get("path") or record.get("image_path") or "unknown")
    stem = path.stem
    return stem.split("_slice", 1)[0] if "_slice" in stem else stem


def image_for_display(image):
    image = image.detach().cpu()
    if image.ndim == 3 and image.shape[0] == 1:
        return image[0].numpy()
    if image.ndim == 3 and image.shape[0] >= 3:
        channels = image[:3]
        if torch.allclose(channels[0], channels[1]) and torch.allclose(channels[1], channels[2]):
            return channels[0].numpy()
        return channels.permute(1, 2, 0).numpy()
    if image.ndim == 2:
        return image.numpy()
    raise ValueError(f"Unsupported image shape: {tuple(image.shape)}")


def update_confusion(confusion, preds, targets):
    pred_flat = preds.detach().cpu().numpy().astype(np.int64).ravel()
    target_flat = targets.detach().cpu().numpy().astype(np.int64).ravel()
    valid = (target_flat >= 0) & (target_flat < NUM_CLASSES) & (pred_flat >= 0) & (pred_flat < NUM_CLASSES)
    encoded = NUM_CLASSES * target_flat[valid] + pred_flat[valid]
    confusion += np.bincount(encoded, minlength=NUM_CLASSES * NUM_CLASSES).reshape(NUM_CLASSES, NUM_CLASSES)


def binary_surface_distances(pred_mask, target_mask):
    pred_mask = pred_mask.astype(bool)
    target_mask = target_mask.astype(bool)
    if not pred_mask.any() or not target_mask.any():
        return np.nan, np.nan
    pred_surface = pred_mask ^ ndi.binary_erosion(pred_mask)
    target_surface = target_mask ^ ndi.binary_erosion(target_mask)
    if not pred_surface.any() or not target_surface.any():
        return np.nan, np.nan
    distance_to_pred = ndi.distance_transform_edt(~pred_surface)
    distance_to_target = ndi.distance_transform_edt(~target_surface)
    distances = np.concatenate([distance_to_pred[target_surface], distance_to_target[pred_surface]])
    if distances.size == 0:
        return np.nan, np.nan
    return float(np.percentile(distances, 95)), float(distances.mean())


def confusion_to_metrics(confusion, model_name):
    tp = np.diag(confusion).astype(np.float64)
    fp = confusion.sum(axis=0) - tp
    fn = confusion.sum(axis=1) - tp
    total = confusion.sum()
    tn = total - tp - fp - fn
    support = confusion.sum(axis=1)
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    f1 = safe_divide(2 * precision * recall, precision + recall)
    dice = safe_divide(2 * tp, 2 * tp + fp + fn)
    iou = safe_divide(tp, tp + fp + fn)
    return pd.DataFrame({
        "model": model_name,
        "class_id": np.arange(NUM_CLASSES),
        "class_name": CLASS_NAMES,
        "support_pixels": support.astype(np.int64),
        "tp": tp.astype(np.int64),
        "fp": fp.astype(np.int64),
        "fn": fn.astype(np.int64),
        "tn": tn.astype(np.int64),
        "precision": precision,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "f1": f1,
        "dice": dice,
        "iou": iou,
    })


def add_average_rows(class_df, model_name):
    fg = class_df[class_df["class_id"] > 0].copy()
    metric_cols = ["precision", "recall_sensitivity", "specificity", "f1", "dice", "iou", "roc_auc", "average_precision", "pr_auc", "hd95", "asd"]
    rows = [{"model": model_name, "average": "foreground_macro"}, {"model": model_name, "average": "foreground_weighted"}]
    support_sum = fg["support_pixels"].sum()
    for col in metric_cols:
        if col in fg.columns:
            rows[0][col] = float(np.nanmean(fg[col].values))
            rows[1][col] = float(np.nansum(fg[col].values * fg["support_pixels"].values) / support_sum) if support_sum else np.nan
    sums = fg[["tp", "fp", "fn", "tn"]].sum()
    micro_precision = safe_divide(sums["tp"], sums["tp"] + sums["fp"]).item()
    micro_recall = safe_divide(sums["tp"], sums["tp"] + sums["fn"]).item()
    rows.append({
        "model": model_name,
        "average": "foreground_micro",
        "precision": micro_precision,
        "recall_sensitivity": micro_recall,
        "f1": safe_divide(2 * micro_precision * micro_recall, micro_precision + micro_recall).item(),
        "dice": safe_divide(2 * sums["tp"], 2 * sums["tp"] + sums["fp"] + sums["fn"]).item(),
        "iou": safe_divide(sums["tp"], sums["tp"] + sums["fp"] + sums["fn"]).item(),
    })
    return pd.DataFrame(rows)


def compute_curve_metrics(y_true, y_score):
    rows, roc_curves, pr_curves = [], {}, {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        y_binary = (y_true == class_id).astype(np.int32)
        scores = y_score[:, class_id]
        if np.unique(y_binary).size < 2:
            rows.append({"class_id": class_id, "class_name": class_name, "roc_auc": np.nan, "average_precision": np.nan, "pr_auc": np.nan})
            continue
        fpr, tpr, _ = roc_curve(y_binary, scores)
        precision, recall, _ = precision_recall_curve(y_binary, scores)
        roc_auc = auc(fpr, tpr)
        ap = average_precision_score(y_binary, scores)
        pr_auc = auc(recall[::-1], precision[::-1])
        roc_curves[class_id] = {"fpr": fpr, "tpr": tpr, "auc": roc_auc}
        pr_curves[class_id] = {"precision": precision, "recall": recall, "ap": ap, "auc": pr_auc}
        rows.append({"class_id": class_id, "class_name": class_name, "roc_auc": roc_auc, "average_precision": ap, "pr_auc": pr_auc})
    y_onehot = (y_true[:, None] == np.arange(NUM_CLASSES)).astype(np.int32)
    if np.unique(y_onehot.ravel()).size == 2:
        fpr, tpr, _ = roc_curve(y_onehot.ravel(), y_score.ravel())
        precision, recall, _ = precision_recall_curve(y_onehot.ravel(), y_score.ravel())
        roc_curves["micro"] = {"fpr": fpr, "tpr": tpr, "auc": auc(fpr, tpr)}
        pr_curves["micro"] = {
            "precision": precision,
            "recall": recall,
            "ap": average_precision_score(y_onehot.ravel(), y_score.ravel()),
            "auc": auc(recall[::-1], precision[::-1]),
        }
    return pd.DataFrame(rows), roc_curves, pr_curves


def slice_dice_rows(preds, targets, start_index, model_name):
    pred_np = preds.detach().cpu().numpy().astype(np.int64)
    target_np = targets.detach().cpu().numpy().astype(np.int64)
    rows = []
    for batch_index in range(pred_np.shape[0]):
        record_index = start_index + batch_index
        record = records["test"][record_index] if record_index < len(records["test"]) else {}
        row = {"model": model_name, "sample_index": record_index, "case_id": case_id_from_record(record)}
        dice_values = []
        for class_id in range(1, NUM_CLASSES):
            pred_mask = pred_np[batch_index] == class_id
            target_mask = target_np[batch_index] == class_id
            denom = pred_mask.sum() + target_mask.sum()
            dice = np.nan if denom == 0 else (2.0 * np.logical_and(pred_mask, target_mask).sum()) / denom
            row[f"dice_class_{class_id}"] = dice
            if not np.isnan(dice):
                dice_values.append(dice)
        row["foreground_dice_mean"] = float(np.nanmean(dice_values)) if dice_values else np.nan
        rows.append(row)
    return rows


@torch.no_grad()
def evaluate_model_detailed(model_name, model):
    criterion = DiceCrossEntropyLoss()
    confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    loss_total = 0.0
    batch_count = 0
    sample_index = 0
    slice_rows = []
    surface_values = {class_id: {"hd95": [], "asd": []} for class_id in range(1, NUM_CLASSES)}
    y_true_chunks, y_score_chunks = [], []
    rng = np.random.default_rng(42)
    total_pixel_count = len(loaders["test"].dataset) * IMAGE_SIZE[0] * IMAGE_SIZE[1]
    max_surface_slices = len(loaders["test"].dataset) if MAX_SURFACE_SLICES <= 0 else MAX_SURFACE_SLICES

    for images, masks in loaders["test"]:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        logits = resize_logits_to_target(model(images), masks)
        loss = criterion(logits, masks)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)
        loss_total += float(loss.detach().cpu())
        batch_count += 1
        update_confusion(confusion, preds, masks)
        slice_rows.extend(slice_dice_rows(preds, masks, sample_index, model_name))

        probs_np = probs.detach().cpu().permute(0, 2, 3, 1).numpy().reshape(-1, NUM_CLASSES)
        targets_np = masks.detach().cpu().numpy().reshape(-1)
        if MAX_ROC_PIXELS > 0:
            if total_pixel_count <= MAX_ROC_PIXELS:
                indices = np.arange(targets_np.shape[0])
            else:
                take = min(targets_np.shape[0], max(1, int(round(targets_np.shape[0] * MAX_ROC_PIXELS / total_pixel_count))))
                indices = rng.choice(targets_np.shape[0], size=take, replace=False)
            y_true_chunks.append(targets_np[indices])
            y_score_chunks.append(probs_np[indices].astype(np.float32))

        pred_np = preds.detach().cpu().numpy().astype(np.int64)
        mask_np = masks.detach().cpu().numpy().astype(np.int64)
        for batch_index in range(pred_np.shape[0]):
            if sample_index + batch_index >= max_surface_slices:
                continue
            for class_id in range(1, NUM_CLASSES):
                hd95, asd = binary_surface_distances(pred_np[batch_index] == class_id, mask_np[batch_index] == class_id)
                if not np.isnan(hd95):
                    surface_values[class_id]["hd95"].append(hd95)
                if not np.isnan(asd):
                    surface_values[class_id]["asd"].append(asd)
        sample_index += masks.shape[0]

    class_df = confusion_to_metrics(confusion, model_name)
    class_df["loss"] = loss_total / max(1, batch_count)
    if y_true_chunks:
        y_true = np.concatenate(y_true_chunks)
        y_score = np.concatenate(y_score_chunks)
        if y_true.shape[0] > MAX_ROC_PIXELS:
            indices = rng.choice(y_true.shape[0], size=MAX_ROC_PIXELS, replace=False)
            y_true, y_score = y_true[indices], y_score[indices]
        curve_df, roc_curves, pr_curves = compute_curve_metrics(y_true, y_score)
        class_df = class_df.merge(curve_df, on=["class_id", "class_name"], how="left")
    else:
        roc_curves, pr_curves = {}, {}
        class_df[["roc_auc", "average_precision", "pr_auc"]] = np.nan

    for class_id in range(1, NUM_CLASSES):
        class_df.loc[class_df["class_id"] == class_id, "hd95"] = float(np.nanmean(surface_values[class_id]["hd95"])) if surface_values[class_id]["hd95"] else np.nan
        class_df.loc[class_df["class_id"] == class_id, "asd"] = float(np.nanmean(surface_values[class_id]["asd"])) if surface_values[class_id]["asd"] else np.nan
    class_df.loc[class_df["class_id"] == 0, ["hd95", "asd"]] = np.nan

    average_df = add_average_rows(class_df, model_name)
    summary = {
        "model": model_name,
        "loss": loss_total / max(1, batch_count),
        "pixel_accuracy": float(np.trace(confusion) / max(1, confusion.sum())),
        "foreground_mean_dice": float(np.nanmean(class_df.loc[class_df["class_id"] > 0, "dice"])),
        "foreground_mean_iou": float(np.nanmean(class_df.loc[class_df["class_id"] > 0, "iou"])),
        "foreground_mean_f1": float(np.nanmean(class_df.loc[class_df["class_id"] > 0, "f1"])),
        "foreground_macro_roc_auc": float(np.nanmean(class_df.loc[class_df["class_id"] > 0, "roc_auc"])),
        "foreground_macro_average_precision": float(np.nanmean(class_df.loc[class_df["class_id"] > 0, "average_precision"])),
        "foreground_macro_pr_auc": float(np.nanmean(class_df.loc[class_df["class_id"] > 0, "pr_auc"])),
        "foreground_mean_hd95": float(np.nanmean(class_df.loc[class_df["class_id"] > 0, "hd95"])),
        "foreground_mean_asd": float(np.nanmean(class_df.loc[class_df["class_id"] > 0, "asd"])),
    }
    if "micro" in roc_curves:
        summary["micro_roc_auc"] = float(roc_curves["micro"]["auc"])
    if "micro" in pr_curves:
        summary["micro_average_precision"] = float(pr_curves["micro"]["ap"])
        summary["micro_pr_auc"] = float(pr_curves["micro"]["auc"])
    return {
        "summary": summary,
        "class_metrics": class_df,
        "average_metrics": average_df,
        "confusion": confusion,
        "slice_metrics": pd.DataFrame(slice_rows),
        "roc_curves": roc_curves,
        "pr_curves": pr_curves,
    }

In [ ]:
results = {}
for model_name, artifact in model_artifacts.items():
    cfg = MODEL_CONFIGS[model_name]
    print(f"\nEvaluating {model_name}...")
    model = build_model(model_name, cfg, artifact["model_info"]).to(DEVICE)
    checkpoint = load_checkpoint(artifact["checkpoint_path"], model=model, map_location=DEVICE)
    artifact["checkpoint_keys"] = sorted(checkpoint.keys()) if isinstance(checkpoint, dict) else []
    model.eval()
    print(f"Loaded checkpoint: {artifact['checkpoint_path']}")
    print(f"Trainable params at eval graph: {count_parameters(model):,}")
    results[model_name] = evaluate_model_detailed(model_name, model)
    display(pd.DataFrame([results[model_name]["summary"]]).T)
    del checkpoint, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metadata_df = pd.DataFrame([
    {
        "model": model_name,
        "architecture": artifact["architecture"],
        "variation": artifact["variation"],
        "contrastive_label": artifact["contrastive_label"],
        "contrastive_weight": artifact["contrastive_weight"],
        "handle": artifact["handle"],
        "checkpoint_path": artifact["checkpoint_path"],
    }
    for model_name, artifact in model_artifacts.items()
])

summary_df = pd.DataFrame([result["summary"] for result in results.values()]).merge(metadata_df, on="model", how="left")
class_metrics_df = pd.concat([result["class_metrics"] for result in results.values()], ignore_index=True).merge(metadata_df, on="model", how="left")
average_metrics_df = pd.concat([result["average_metrics"] for result in results.values()], ignore_index=True).merge(metadata_df, on="model", how="left")
slice_metrics_df = pd.concat([result["slice_metrics"] for result in results.values()], ignore_index=True).merge(metadata_df, on="model", how="left")

comparison_metrics = ["dice", "iou", "f1", "roc_auc", "average_precision", "pr_auc", "hd95", "asd"]
metric_cols = [metric for metric in comparison_metrics if metric in class_metrics_df.columns]
variant_df = class_metrics_df[class_metrics_df["class_id"] > 0].copy()
baseline_df = variant_df[variant_df["contrastive_weight"] == 0][["architecture", "class_id", "model"] + metric_cols].copy()
baseline_df = baseline_df.rename(columns={"model": "baseline_model", **{metric: f"{metric}_baseline" for metric in metric_cols}})
delta_df = variant_df.merge(baseline_df, on=["architecture", "class_id"], how="left")
for metric in metric_cols:
    delta_df[f"delta_{metric}"] = delta_df[metric] - delta_df[f"{metric}_baseline"]

summary_df.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)
class_metrics_df.to_csv(OUTPUT_DIR / "class_metrics.csv", index=False)
average_metrics_df.to_csv(OUTPUT_DIR / "average_metrics.csv", index=False)
slice_metrics_df.to_csv(OUTPUT_DIR / "slice_metrics.csv", index=False)
delta_df.to_csv(OUTPUT_DIR / "model_delta_vs_cw0_metrics.csv", index=False)
(OUTPUT_DIR / "model_artifacts.json").write_text(json.dumps(model_artifacts, indent=2, ensure_ascii=False), encoding="utf-8")

print("Summary metrics")
display(summary_df.sort_values(["architecture", "contrastive_weight"]))
print("Per-class metrics")
display(class_metrics_df)
print("Average metrics")
display(average_metrics_df)
print("Delta metrics vs each architecture's cw0 baseline")
display(delta_df)
print(f"Saved CSV reports to: {OUTPUT_DIR}")

In [ ]:
def save_current_figure(name):
    path = OUTPUT_DIR / name
    plt.savefig(path, dpi=160, bbox_inches="tight")
    print(f"Saved figure: {path}")


def plot_summary_table(summary):
    display_cols = [
        "model", "loss", "pixel_accuracy", "foreground_mean_dice", "foreground_mean_iou",
        "foreground_mean_f1", "foreground_macro_roc_auc", "foreground_macro_average_precision",
        "micro_roc_auc", "micro_average_precision", "foreground_mean_hd95", "foreground_mean_asd",
    ]
    table_df = summary[[col for col in display_cols if col in summary.columns]].copy()
    numeric_cols = table_df.select_dtypes(include=[np.number]).columns
    table_df[numeric_cols] = table_df[numeric_cols].round(4)
    fig, ax = plt.subplots(figsize=(15, 2.6))
    ax.axis("off")
    table = ax.table(cellText=table_df.values, colLabels=table_df.columns, loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.35)
    ax.set_title("Model summary metrics", fontsize=14, pad=14)
    save_current_figure("summary_table.png")
    plt.show()


def plot_model_auc_summary(summary):
    auc_cols = [
        "foreground_macro_roc_auc",
        "micro_roc_auc",
        "foreground_macro_average_precision",
        "micro_average_precision",
        "foreground_macro_pr_auc",
        "micro_pr_auc",
    ]
    available = [col for col in auc_cols if col in summary.columns]
    if not available:
        print("No ROC-AUC/AP columns available for model-level AUC summary.")
        return
    plot_df = summary[["model"] + available].melt(id_vars="model", var_name="metric", value_name="value")
    plot_df = plot_df.dropna(subset=["value"])
    if plot_df.empty:
        print("No valid ROC-AUC/AP values available for model-level AUC summary.")
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=plot_df, x="metric", y="value", hue="model", ax=ax)
    ax.set_title("Model-level ROC-AUC and PR-AUC/AP summary")
    ax.set_xlabel("")
    ax.set_ylabel("score")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
    ax.legend(loc="best")
    save_current_figure("model_auc_summary.png")
    plt.show()


def plot_metric_bars(metrics_df, metric, title, filename):
    plot_df = metrics_df[metrics_df["class_id"] > 0].copy()
    plot_df = plot_df.dropna(subset=[metric])
    if plot_df.empty:
        print(f"No valid values available for {metric}.")
        return
    fig, ax = plt.subplots(figsize=(13, 5))
    sns.barplot(data=plot_df, x="class_name", y=metric, hue="model", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=35)
    if metric not in {"hd95", "asd"}:
        ax.set_ylim(0, 1)
    ax.legend(loc="best")
    save_current_figure(filename)
    plt.show()


def plot_confusion(confusion, model_name, normalize=False):
    if normalize:
        matrix = safe_divide(confusion.astype(np.float64), confusion.sum(axis=1, keepdims=True))
        fmt = ".2f"
    else:
        matrix = confusion.astype(np.int64)
        fmt = "d"
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        matrix,
        annot=True,
        fmt=fmt,
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax,
        cbar=True,
    )
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"{model_name} confusion matrix" + (" normalized" if normalize else ""))
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
    suffix = "normalized" if normalize else "raw"
    save_current_figure(f"confusion_{model_name.lower().replace('-', '_')}_{suffix}.png")
    plt.show()


def plot_curves(curves_by_model, curve_type):
    for model_name, curves in curves_by_model.items():
        if not curves:
            print(f"No valid {curve_type.upper()} curve for {model_name}.")
            continue
        fig, ax = plt.subplots(figsize=(8, 7))
        for key, curve in curves.items():
            if key == "micro":
                continue
            if curve_type == "roc":
                ax.plot(curve["fpr"], curve["tpr"], linewidth=1.4, label=f"{CLASS_NAMES[key]} ({curve['auc']:.3f})")
            else:
                ax.plot(curve["recall"], curve["precision"], linewidth=1.4, label=f"{CLASS_NAMES[key]} AP={curve['ap']:.3f}")
        if "micro" in curves:
            curve = curves["micro"]
            if curve_type == "roc":
                ax.plot(curve["fpr"], curve["tpr"], color="black", linestyle="--", linewidth=2, label=f"micro ({curve['auc']:.3f})")
            else:
                ax.plot(curve["recall"], curve["precision"], color="black", linestyle="--", linewidth=2, label=f"micro AP={curve['ap']:.3f}")
        if curve_type == "roc":
            ax.plot([0, 1], [0, 1], color="gray", linestyle=":", linewidth=1)
            ax.set_xlabel("False positive rate")
            ax.set_ylabel("True positive rate")
            ax.set_title(f"{model_name} ROC curves")
        else:
            ax.set_xlabel("Recall")
            ax.set_ylabel("Precision")
            ax.set_title(f"{model_name} Precision-Recall curves")
        ax.legend(loc="best", fontsize=8)
        ax.grid(True, alpha=0.25)
        save_current_figure(f"{curve_type}_curves_{model_name.lower().replace('-', '_')}.png")
        plt.show()


def plot_delta(delta, metric):
    delta_col = f"delta_{metric}"
    plot_df = delta.dropna(subset=[delta_col]).copy()
    if plot_df.empty:
        print(f"No valid delta values available for {metric}.")
        return
    fig, ax = plt.subplots(figsize=(12, 4.8))
    sns.barplot(data=plot_df, x="class_name", y=delta_col, hue="model", ax=ax)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"Delta vs same-architecture cw0 baseline: {metric}")
    ax.set_xlabel("")
    ax.set_ylabel(delta_col)
    ax.tick_params(axis="x", rotation=35)
    ax.legend(loc="best", fontsize=8)
    save_current_figure(f"delta_vs_cw0_{metric}.png")
    plt.show()


def plot_slice_boxplot(slice_df):
    if slice_df.empty or slice_df["foreground_dice_mean"].notna().sum() == 0:
        print("No valid slice-level Dice values available for boxplot.")
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=slice_df, x="model", y="foreground_dice_mean", ax=ax)
    sns.stripplot(data=slice_df.sample(min(len(slice_df), 400), random_state=42), x="model", y="foreground_dice_mean", color="black", alpha=0.25, size=2, ax=ax)
    ax.set_title("Slice-level foreground Dice distribution")
    ax.set_xlabel("")
    ax.set_ylabel("Foreground Dice mean")
    ax.set_ylim(0, 1)
    save_current_figure("slice_dice_boxplot.png")
    plt.show()


plot_summary_table(summary_df)
plot_model_auc_summary(summary_df)
for metric_name in ["dice", "iou", "f1", "roc_auc", "average_precision", "pr_auc", "hd95", "asd"]:
    plot_metric_bars(class_metrics_df, metric_name, f"Per-class {metric_name}", f"bar_{metric_name}.png")

for model_name, result in results.items():
    plot_confusion(result["confusion"], model_name, normalize=False)
    plot_confusion(result["confusion"], model_name, normalize=True)

plot_curves({name: result["roc_curves"] for name, result in results.items()}, "roc")
plot_curves({name: result["pr_curves"] for name, result in results.items()}, "pr")

for metric_name in ["dice", "iou", "f1", "roc_auc", "average_precision", "hd95", "asd"]:
    plot_delta(delta_df, metric_name)
plot_slice_boxplot(slice_metrics_df)

In [ ]:
@torch.no_grad()
def show_model_comparison_predictions(model_names, loader, max_samples=4, prefer_foreground=True):
    cmap = ListedColormap(plt.get_cmap("tab20", NUM_CLASSES)(np.arange(NUM_CLASSES)))
    samples = []
    for images, masks in loader:
        for idx in range(images.shape[0]):
            if prefer_foreground and not (masks[idx] > 0).any():
                continue
            samples.append((images[idx], masks[idx]))
            if len(samples) >= max_samples:
                break
        if len(samples) >= max_samples:
            break
    if not samples:
        images, masks = next(iter(loader))
        samples = [(images[idx], masks[idx]) for idx in range(min(max_samples, images.shape[0]))]

    predictions = {model_name: [] for model_name in model_names}
    for model_name in model_names:
        artifact = model_artifacts[model_name]
        cfg = MODEL_CONFIGS[model_name]
        model = build_model(model_name, cfg, artifact["model_info"]).to(DEVICE)
        load_checkpoint(artifact["checkpoint_path"], model=model, map_location=DEVICE)
        model.eval()
        for image, mask in samples:
            input_tensor = image.unsqueeze(0).to(DEVICE)
            logits = resize_logits_to_target(model(input_tensor), mask.unsqueeze(0).to(DEVICE))
            predictions[model_name].append(logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.int64))
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    columns = ["Image", "Ground truth"]
    for model_name in model_names:
        columns.extend([f"{model_name} pred", f"{model_name} overlay"])
    fig, axes = plt.subplots(len(samples), len(columns), figsize=(4.2 * len(columns), 4 * len(samples)))
    if len(samples) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_index, (image, mask) in enumerate(samples):
        image_np = image_for_display(image)
        mask_np = mask.detach().cpu().numpy().astype(np.int64)
        axes[row_index, 0].imshow(image_np, cmap="gray")
        axes[row_index, 0].set_title("Image")
        axes[row_index, 0].axis("off")
        axes[row_index, 1].imshow(mask_np, cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1)
        axes[row_index, 1].set_title("Ground truth")
        axes[row_index, 1].axis("off")
        col = 2
        for model_name in model_names:
            pred_np = predictions[model_name][row_index]
            axes[row_index, col].imshow(pred_np, cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1)
            axes[row_index, col].set_title(f"{model_name} pred")
            axes[row_index, col].axis("off")
            axes[row_index, col + 1].imshow(image_np, cmap="gray")
            axes[row_index, col + 1].imshow(mask_np, cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1, alpha=0.28)
            axes[row_index, col + 1].imshow(pred_np, cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1, alpha=0.45)
            axes[row_index, col + 1].set_title(f"{model_name} overlay")
            axes[row_index, col + 1].axis("off")
            col += 2
    plt.tight_layout()
    save_current_figure("prediction_comparison_demo.png")
    plt.show()

prediction_model_names = summary_df.sort_values("foreground_mean_dice", ascending=False)["model"].head(MAX_PREDICTION_MODELS).tolist()
print("Prediction demo models:", prediction_model_names)
show_model_comparison_predictions(prediction_model_names, loaders["test"], max_samples=4, prefer_foreground=True)

In [ ]:
report = {
    "data_root": str(DATA_ROOT),
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "max_roc_pixels": MAX_ROC_PIXELS,
    "max_surface_slices": MAX_SURFACE_SLICES or "all",
    "model_artifacts": model_artifacts,
    "summary": summary_df.to_dict(orient="records"),
}
(OUTPUT_DIR / "evaluation_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Evaluation report saved to: {OUTPUT_DIR}")
print("Files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)